# 03 — Story generation

**Project:** [EmoVecLLM](https://github.com/drgzkr/EmoVecLLM) — replicating Anthropic 2026 emotion vectors on open-source LLMs.

This notebook turns the prompt manifest from **nb02** into the emotion-conditioned passages (and the neutral-dialogue baseline) that nb04 will run activations on.

**What it does**

1. Resolve a portable `WORK_DIR` (Google Drive on Colab, `EMOVEC_WORK_DIR` on HPC/local) and load `dataset_spec.json` + `prompts.jsonl`.
2. Load a **swappable** generator (`GENERATOR_MODEL` from the spec, or `EMOVEC_GENERATOR_MODEL`) at a precision that fits the GPU — bf16/fp16 on a big card, 4-bit (bitsandbytes) on a T4, `device_map="auto"` across multiple GPUs on a cluster.
3. Decode the prompt jobs in **batches**, parse each multi-item completion into individual stories/dialogues (keeping the raw text), and rewrite `Person:`/`AI:` → `Human:`/`Assistant:` in dialogues per Anthropic.
4. Append results to `data/processed/stories/{spec_hash}/{model}/stories.jsonl`, **skipping jobs already done** so the run resumes after any interruption.

**Designed to run head-less.** Every knob has an environment-variable override, so on an HPC node you can `EMOVEC_WORK_DIR=… EMOVEC_GENERATOR_MODEL=… jupyter nbconvert --to notebook --execute 03_story_generation.ipynb` (or drive it with `papermill`) without editing a cell.

## 0 — Setup

Same portable bootstrap as nb02, plus the generation stack (`torch`, `transformers`, `accelerate`, `bitsandbytes`).

| Env var | Effect | Default |
|---|---|---|
| `EMOVEC_WORK_DIR` | Where prompts/stories live. | Drive on Colab, repo root locally |
| `EMOVEC_MOUNT_DRIVE` | Mount Google Drive on Colab. | `1` on Colab |
| `HF_TOKEN` | HF token for gated models (Llama-3). | Colab Secrets → env |
| `EMOVEC_GENERATOR_MODEL` | Override the generator from the spec. | spec value |
| `EMOVEC_PRECISION` | `auto`/`bf16`/`fp16`/`4bit`/`8bit`. | `auto` |
| `EMOVEC_DEVICE_MAP` | HF device map (`auto` spreads GPUs). | `auto` |
| `EMOVEC_BATCH_SIZE` | Prompts decoded together. | `8` |
| `EMOVEC_MAX_JOBS` | Cap jobs this run (`0` = all). | `0` |
| `EMOVEC_KINDS` | Restrict kinds, csv. | all |
| `EMOVEC_SMOKE_TEST` | Tiny run to check wiring. | `0` |
| `EMOVEC_RESUME` | Skip already-done job_ids. | `1` |

In [ ]:
# Dependencies. Installed on Colab; assumed present on a provisioned HPC env.
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.40", "accelerate>=0.30",
                    "bitsandbytes>=0.43", "tqdm"], check=False)


In [ ]:
import json, os, time, hashlib, re, random
from pathlib import Path
from collections import Counter

def _env_flag(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ("1", "true", "yes", "on")

# ── Hugging Face token (gated models). env var > Colab Secrets. ─────────────
if "HF_TOKEN" not in os.environ and IN_COLAB:
    try:
        from google.colab import userdata
        _t = userdata.get("HF_TOKEN")
        if _t:
            os.environ["HF_TOKEN"] = _t
    except Exception:
        pass

# ── Working directory (must match nb02 so we find prompts.jsonl) ────────────
MOUNT_DRIVE = _env_flag("EMOVEC_MOUNT_DRIVE", IN_COLAB)
if os.environ.get("EMOVEC_WORK_DIR"):
    WORK_DIR = Path(os.environ["EMOVEC_WORK_DIR"]).expanduser()
elif IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/EmoVecLLM")
elif IN_COLAB:
    WORK_DIR = Path("/content/EmoVecLLM")
else:
    WORK_DIR = Path("..").resolve()

DATA_DIR = WORK_DIR / "data" / "processed"
DATA_DIR.mkdir(parents=True, exist_ok=True)

import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    N_GPU = torch.cuda.device_count()
else:
    DEVICE, GPU_NAME, VRAM_GB, N_GPU = "cpu", "cpu", 0.0, 0

print(f"IN_COLAB={IN_COLAB}  WORK_DIR={WORK_DIR}")
print(f"device  : {DEVICE}  ({GPU_NAME}, {VRAM_GB:.0f} GB × {N_GPU} GPU)")
print(f"HF_TOKEN set: {'HF_TOKEN' in os.environ}")


## 1 — Generation config

The only line you usually touch is `GENERATOR_MODEL_OVERRIDE` (or set `EMOVEC_GENERATOR_MODEL`). Everything else auto-adapts to the hardware, but is overridable for a cluster run.

In [ ]:
def _env(name, default):
    v = os.environ.get(name)
    return v if v not in (None, "") else default

def _env_int(name, default):
    v = os.environ.get(name)
    return int(v) if v not in (None, "") else default

# Which generator? "" → use whatever nb02 baked into the spec. Swap freely:
#   "Qwen/Qwen2.5-7B-Instruct"  (default, Apache, no gating)
#   "meta-llama/Llama-3.1-8B-Instruct"  (needs HF_TOKEN + license)
#   "mistralai/Mistral-7B-Instruct-v0.3"
#   "EleutherAI/pythia-1.4b"  (base model; rough prose, fast prototyping)
GENERATOR_MODEL_OVERRIDE = _env("EMOVEC_GENERATOR_MODEL", "")

PRECISION  = _env("EMOVEC_PRECISION", "auto")     # auto|bf16|fp16|4bit|8bit
DEVICE_MAP = _env("EMOVEC_DEVICE_MAP", "auto")    # "auto" spreads across GPUs

BATCH_SIZE  = _env_int("EMOVEC_BATCH_SIZE", 8)    # prompts decoded together
MAX_JOBS    = _env_int("EMOVEC_MAX_JOBS", 0)      # 0 → all pending jobs
KIND_FILTER = _env("EMOVEC_KINDS", "")            # csv subset of kinds; "" = all
SMOKE_TEST  = _env_flag("EMOVEC_SMOKE_TEST", False)
RESUME      = _env_flag("EMOVEC_RESUME", True)

if SMOKE_TEST:
    MAX_JOBS = MAX_JOBS or 4
    BATCH_SIZE = min(BATCH_SIZE, 2)

print(f"model override : {GENERATOR_MODEL_OVERRIDE or '(use spec)'}")
print(f"precision      : {PRECISION}    device_map: {DEVICE_MAP}")
print(f"batch_size     : {BATCH_SIZE}    max_jobs: {MAX_JOBS or 'all'}    smoke: {SMOKE_TEST}")
print(f"kind_filter    : {KIND_FILTER or 'all'}    resume: {RESUME}")


## 2 — Load the spec and prompt manifest

Both files come from nb02. The decoding parameters (temperature, top-p, target tokens) are read from the spec so generation matches what was frozen there.

In [ ]:
spec_path    = DATA_DIR / "dataset_spec.json"
prompts_path = DATA_DIR / "prompts.jsonl"
assert spec_path.exists(),    f"missing {spec_path} — run nb02 first"
assert prompts_path.exists(), f"missing {prompts_path} — run nb02 first"

spec = json.loads(spec_path.read_text())
SPEC_HASH       = spec["spec_hash"]
GENERATOR_MODEL = GENERATOR_MODEL_OVERRIDE or spec["generator_model"]

dec = spec["decoding"]
TEMPERATURE    = dec["temperature"]
TOP_P          = dec["top_p"]
MAX_NEW_TOKENS = _env_int("EMOVEC_MAX_NEW_TOKENS", dec["target_tokens_per_call"])
USER_TRIGGER   = spec["prompts"]["user_trigger"]

jobs = [json.loads(l) for l in prompts_path.read_text().splitlines() if l.strip()]
if KIND_FILTER:
    keep = {k.strip() for k in KIND_FILTER.split(",")}
    jobs = [j for j in jobs if j["kind"] in keep]

print(f"spec_hash    : {SPEC_HASH}")
print(f"generator    : {GENERATOR_MODEL}")
print(f"decoding     : T={TEMPERATURE}  top_p={TOP_P}  max_new_tokens={MAX_NEW_TOKENS}")
print(f"jobs loaded  : {len(jobs)}")
for k, c in sorted(Counter(j['kind'] for j in jobs).items()):
    print(f"  {k:<20} {c:>5}")


## 3 — Load the generator (swappable)

`load_generator` is model-agnostic: it works for any chat/instruct `AutoModelForCausalLM`. `resolve_precision("auto", …)` picks **4-bit** on GPUs under ~24 GB (so 7–8B models fit a free T4) and **bf16** otherwise. `device_map="auto"` lets `accelerate` shard a large model across all visible GPUs on an HPC node — the same cell scales from a T4 to a multi-A100 box without edits.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

def resolve_precision(pref, vram_gb):
    if pref != "auto":
        return pref
    if DEVICE == "cpu":
        return "fp32"
    return "4bit" if (vram_gb and vram_gb < 24) else "bf16"

def load_generator(name, precision="auto", device_map="auto"):
    prec = resolve_precision(precision, VRAM_GB)
    hf_token = os.environ.get("HF_TOKEN")

    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True, token=hf_token)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"   # decoder-only batched generation pads on the left

    kw = dict(trust_remote_code=True, token=hf_token)
    if DEVICE == "cpu":
        kw.update(torch_dtype=torch.float32)
    elif prec in ("4bit", "8bit") and BitsAndBytesConfig is not None:
        kw.update(
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=(prec == "4bit"),
                load_in_8bit=(prec == "8bit"),
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            ),
            device_map=device_map,
        )
    else:
        dtype = (torch.bfloat16
                 if prec == "bf16" and torch.cuda.is_bf16_supported()
                 else torch.float16)
        kw.update(torch_dtype=dtype, device_map=device_map)

    model = AutoModelForCausalLM.from_pretrained(name, **kw)
    model.eval()
    return tok, model, prec

print(f"loading {GENERATOR_MODEL} …")
_t0 = time.time()
tokenizer, model, USED_PRECISION = load_generator(GENERATOR_MODEL, PRECISION, DEVICE_MAP)
INPUT_DEVICE = model.get_input_embeddings().weight.device
print(f"loaded in {time.time() - _t0:.0f}s  (precision={USED_PRECISION}, input_device={INPUT_DEVICE})")


## 4 — Generation + parsing helpers

- `build_chat_text` applies the model's chat template, falling back to merging the system message into the user turn for models without a `system` role (e.g. Gemma), and to plain concatenation for base models with no template.
- `generate_batch` left-pads, samples with the spec's temperature/top-p, and returns only the newly generated tokens.
- `split_segments` is a best-effort splitter of a multi-item completion (`[story 1] … [story 2] …`, `1. … 2. …`, or blank-line separated). We **always keep the raw completion** too, so parsing can be revisited later without re-generating.
- `convert_dialogue_roles` rewrites `Person:`/`AI:` → `Human:`/`Assistant:` for dialogue kinds.

In [ ]:
def build_chat_text(system_prompt, user):
    if getattr(tokenizer, "chat_template", None):
        candidates = (
            [{"role": "system", "content": system_prompt},
             {"role": "user",   "content": user}],
            # fallback for templates without a system role
            [{"role": "user", "content": system_prompt + "\n\n" + user}],
        )
        for msgs in candidates:
            try:
                return tokenizer.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=True)
            except Exception:
                continue
    return f"{system_prompt}\n\n{user}\n\n"   # base model, no chat template

@torch.no_grad()
def generate_batch(texts, seeds, max_new_tokens):
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=4096).to(INPUT_DEVICE)
    torch.manual_seed(int(seeds[0]) & 0x7fffffff)   # one RNG per batch
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[:, enc["input_ids"].shape[1]:]
    return tokenizer.batch_decode(gen, skip_special_tokens=True)

# Matches an item header at line start: "[story 1]", "Story 2:", "**Dialogue 3**",
# "Example 4 -", optionally followed by the item text on the same line.
_HEADER_RE = re.compile(
    r"(?im)^[ \t]*\**\[?\s*(?:story|dialogue|example|passage)\s*#?\s*\d+\s*\]?\**[ \t]*[:.)\-]?[ \t]*")
_NUM_RE = re.compile(r"(?im)^[ \t]*\d+[ \t]*[.)][ \t]+")

def split_segments(text):
    text = text.strip()
    for rx in (_HEADER_RE, _NUM_RE):
        parts = [p.strip() for p in rx.split(text) if p.strip()]
        if len(parts) >= 2:
            return parts
    chunks = [c.strip() for c in re.split(r"\n[ \t]*\n", text) if c.strip()]
    return chunks if len(chunks) >= 2 else [text]

def convert_dialogue_roles(text):
    text = re.sub(r"(?im)^[ \t]*Person[ \t]*:", "Human:", text)
    text = re.sub(r"(?im)^[ \t]*AI[ \t]*:", "Assistant:", text)
    return text


## 5 — Output path + resume

Output is keyed by **both** `spec_hash` and the (sanitised) model name, so switching generators never overwrites a previous run. On resume we read the existing `stories.jsonl` and skip any `job_id` already present.

In [ ]:
def _safe(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s)

OUT_DIR = DATA_DIR / "stories" / SPEC_HASH / _safe(GENERATOR_MODEL)
OUT_DIR.mkdir(parents=True, exist_ok=True)
STORIES_PATH = OUT_DIR / "stories.jsonl"

done_ids = set()
if RESUME and STORIES_PATH.exists():
    for line in STORIES_PATH.read_text().splitlines():
        if line.strip():
            try:
                done_ids.add(json.loads(line)["job_id"])
            except Exception:
                pass

pending = [j for j in jobs if j["job_id"] not in done_ids]
if MAX_JOBS:
    pending = pending[:MAX_JOBS]

print(f"output       → {STORIES_PATH}")
print(f"already done : {len(done_ids)}")
print(f"pending now  : {len(pending)}  (of {len(jobs)} total)")


## 6 — Generate

Batched decode with an OOM safety net (falls back to per-item generation if a batch overflows VRAM). Results are flushed to disk after every batch, so an interrupted run loses at most one batch and resumes cleanly.

In [ ]:
from tqdm.auto import tqdm

def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

def _record(job, completion):
    is_dialogue = job["kind"] in ("neutral_dialogue", "emotional_dialogue")
    segs = split_segments(completion)
    if is_dialogue:
        segs = [convert_dialogue_roles(s) for s in segs]
    return {
        "job_id": job["job_id"],
        "kind": job["kind"],
        "emotion": job.get("emotion"),
        "person_emotion": job.get("person_emotion"),
        "ai_emotion": job.get("ai_emotion"),
        "topic": job["topic"],
        "topic_idx": job["topic_idx"],
        "spec_hash": SPEC_HASH,
        "generator_model": GENERATOR_MODEL,
        "n_segments": len(segs),
        "segments": segs,
        "raw": completion,
    }

written, t0 = 0, time.time()
batches = list(chunked(pending, BATCH_SIZE))
with STORIES_PATH.open("a", encoding="utf-8") as fout:
    for batch in tqdm(batches, desc="generating"):
        texts = [build_chat_text(j["system_prompt"], j["user"]) for j in batch]
        seeds = [j["seed"] for j in batch]
        try:
            completions = generate_batch(texts, seeds, MAX_NEW_TOKENS)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            completions = []
            for t, s in zip(texts, seeds):          # retry one-at-a-time
                completions += generate_batch([t], [s], MAX_NEW_TOKENS)
        for job, comp in zip(batch, completions):
            fout.write(json.dumps(_record(job, comp), ensure_ascii=False) + "\n")
            written += 1
        fout.flush()

dt = time.time() - t0
rate = written / max(dt, 1) * 60
print(f"wrote {written} job outputs in {dt/60:.1f} min  ({rate:.1f} jobs/min)")


## 7 — Qualitative spot-check

Two checks on the emotion stories:
1. **Read a few** — do they convey the target emotion?
2. **Leakage** — Anthropic's prompt forbids naming the emotion. We flag segments where the emotion word appears verbatim (a rough proxy; synonyms aren't caught).

In [ ]:
import textwrap

recs = [json.loads(l) for l in STORIES_PATH.read_text().splitlines() if l.strip()]
emo_recs = [r for r in recs if r["kind"] == "emotion_story"]

shown = 0
for r in emo_recs:
    seg = r["segments"][0] if r["segments"] else r["raw"]
    leaked = re.search(rf"\b{re.escape(r['emotion'])}\b", seg, re.I) is not None
    print(f"── {r['emotion']!r}  | topic: {r['topic']}  | leak={leaked} | "
          f"{r['n_segments']} segs ──")
    print(textwrap.fill(seg[:600], 88))
    print()
    shown += 1
    if shown >= 3:
        break
if shown == 0:
    print("no emotion_story outputs yet — run §6 first.")

# leakage rate over all emotion-story segments
total = leaks = 0
for r in emo_recs:
    for seg in r["segments"]:
        total += 1
        if re.search(rf"\b{re.escape(r['emotion'])}\b", seg, re.I):
            leaks += 1
if total:
    print(f"emotion-word leakage: {leaks}/{total} segments ({100*leaks/total:.1f}%)")


## 8 — Summary + next

Write a small `run_manifest.json` next to the stories and report coverage. **nb04** (`04_activation_extraction.ipynb`) will load `stories.jsonl`, run each segment through the *target* model via TransformerLens, mean-pool the residual stream, and stack the activations that nb05 differences into the 171 emotion vectors.

Remember the neutral dialogues are the **baseline** — nb05 takes their top principal components (≈50 % variance) and projects them out of every emotion vector to remove the story-vs-dialogue style direction.

In [ ]:
recs = [json.loads(l) for l in STORIES_PATH.read_text().splitlines() if l.strip()]
by_kind = Counter(r["kind"] for r in recs)
n_seg = sum(r["n_segments"] for r in recs)

run_manifest = {
    "spec_hash": SPEC_HASH,
    "generator_model": GENERATOR_MODEL,
    "precision": USED_PRECISION,
    "stories_file": str(STORIES_PATH),
    "n_job_outputs": len(recs),
    "n_segments_total": n_seg,
    "by_kind": dict(by_kind),
    "n_jobs_in_manifest": len(jobs),
    "updated": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
(OUT_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
print(json.dumps(run_manifest, indent=2))
print()
done = len({r['job_id'] for r in recs})
print(f"coverage: {done}/{len(jobs)} jobs complete "
      f"({100*done/max(len(jobs),1):.1f}%)  |  {n_seg} segments parsed")
